<a href="https://colab.research.google.com/github/spbromberg/Project-3-DS-4002/blob/main/Scripts/01_Paintings_Project_Dataset_Creation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### This Notebook creates a dataset for our analysis

# WikiArt Painting Dataset

This notebook collects a labeled image dataset for the project. It targets **100 paintings in each of six classes** (Renaissance, Baroque, Romanticism, Impressionism, Cubism, and Other), for up to 600 images total. Counts can be lower when WikiArt pages fail or the page metadata does not match the filters.

The collection cell performs the full scrape in one pass. Requests are intentionally delayed, so the collection may take some time. Avoid rerunning the collection cell unless you intend to start another scrape.

The class is inferred from the artist's movement-page listing and checked against each artwork's Style metadata. This is an educational label, not an verified category. Follow WikiArt's terms and respect image copyrights when using or sharing the downloaded files.

### Install dependencies

Install the three external Python packages used to make HTTP requests, parse WikiArt pages, and write the dataset tables. Colab's `google.colab` tools are already available in the runtime.

In [3]:
%pip -q install requests pandas beautifulsoup4

### Imports and settings

Load the libraries, choose the six movement classes, set the per-class goal to 100, and configure a polite two-second delay plus automatic retries for temporary request failures.



In [4]:
import random
import re
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup
from google.colab import files

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_URL = "https://www.wikiart.org"
OUTPUT_FOLDER = Path("/content/wikiart_project")
IMAGE_FOLDER = OUTPUT_FOLDER / "images"
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
IMAGE_FOLDER.mkdir(parents=True, exist_ok=True)

MOVEMENT_PAGES = {
    "Renaissance": [
        "early-renaissance",
        "high-renaissance",
        "northern-renaissance",
        "mannerism-late-renaissance",
    ],
    "Baroque": ["baroque"],
    "Romanticism": ["romanticism"],
    "Impressionism": ["impressionism"],
    "Cubism": ["cubism"],
    "Other": ["surrealism", "realism"],
}

PAINTING_MEDIA_KEYWORDS = [
    "oil", "acrylic", "watercolor", "watercolour", "tempera",
    "gouache", "fresco", "encaustic", "pastel", "casein",
]

TARGET_PER_MOVEMENT = 100
MAX_ARTISTS_PER_MOVEMENT = 50
MAX_ARTWORKS_PER_ARTIST = 15
MAX_ARTWORK_ATTEMPTS_PER_ARTIST = 40
REQUEST_DELAY_SECONDS = 2
RANDOM_SEED = 42

def normalize_label(value):
    return re.sub(r"[^a-z0-9]", "", str(value).lower())

STYLE_TO_MOVEMENT = {
    normalize_label(page_slug.replace("-", " ")): movement
    for movement, page_slugs in MOVEMENT_PAGES.items()
    for page_slug in page_slugs
}

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; educational WikiArt dataset project)"
})
retry_strategy = Retry(
    total=4,
    connect=4,
    read=4,
    status=4,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]),
    respect_retry_after_header=True,
)
session.mount("https://", HTTPAdapter(max_retries=retry_strategy))

### Find and select artists

Visit the selected movement pages once, collect their artist names and page slugs, then make a reproducible pool of up to 50 artists for each class. Saving this list makes it clear which artists were available to the image-collection pass.

In [ ]:
# Read artist candidates from each movement page, remove duplicate artists,
# and select a reproducible pool of up to 50 artists per class. The full pool
# gives the collection pass more chances to reach the 100-image target.

def fetch_wikiart(url):
    """Fetch one WikiArt page or image with a delay and explicit HTTP errors."""
    parsed = urlparse(url)
    host = (parsed.hostname or "").lower()
    if parsed.scheme != "https" or (host != "wikiart.org" and not host.endswith(".wikiart.org")):
        raise ValueError(f"Refusing a non-WikiArt URL: {url}")

    time.sleep(REQUEST_DELAY_SECONDS)
    response = session.get(url, timeout=45)
    response.raise_for_status()
    return response

def get_artist_candidates(movement, page_slugs):
    """Read artist names and slugs from the movement text-list pages."""
    found = {}
    for page_slug in page_slugs:
        page_url = f"{BASE_URL}/en/artists-by-art-movement/{page_slug}/text-list"
        soup = BeautifulSoup(fetch_wikiart(page_url).text, "html.parser")
        for link in soup.select("main .masonry-text-view-all a[href]"):
            artist_url = urljoin(BASE_URL, link["href"])
            parts = [part for part in urlparse(artist_url).path.split("/") if part]
            if len(parts) != 2 or parts[0].lower() != "en":
                continue

            artist_slug = parts[1].lower()
            artist_name = re.sub(r"\s+", " ", link.get_text(" ", strip=True))
            if not artist_name or artist_slug in found:
                continue
            found[artist_slug] = {
                "artist_name": artist_name,
                "artist_slug": artist_slug,
                "movement": movement,
            }
    return list(found.values())

artist_rows = []
for movement, page_slugs in MOVEMENT_PAGES.items():
    candidates = get_artist_candidates(movement, page_slugs)
    if not candidates:
        raise ValueError(f"No artists were found for movement {movement!r}.")
    artist_rows.extend(candidates)
    print(f"{movement}: found {len(candidates)} artist candidates")

artist_candidates = pd.DataFrame(artist_rows).drop_duplicates(
    subset=["artist_slug"], keep="first"
)
selected_groups = []
for movement, group in artist_candidates.groupby("movement", sort=False):
    selected_groups.append(
        group.sort_values("artist_slug").sample(
            n=min(len(group), MAX_ARTISTS_PER_MOVEMENT),
            random_state=RANDOM_SEED,
        )
    )

selected_artists = pd.concat(selected_groups, ignore_index=True)
artists_path = OUTPUT_FOLDER / "artists.csv"
selected_artists.to_csv(artists_path, index=False)
print(f"Selected {len(selected_artists)} artists; saved {artists_path}")
print(selected_artists.groupby("movement").size())

### Define collection helpers

Create reusable functions to find an artist's artwork links, read the artwork's image URL and metadata, keep only records whose Style and medium match the selected class, and save accepted image files. This cell defines functions but does not make requests until the next cell.

In [5]:
# These functions find artwork pages, read the image/style/medium metadata,
# filter for paintings, and save each accepted image under a stable filename.

def get_artwork_links(artist_slug):
    """Find artwork links on an artist's WikiArt works page."""
    page_url = f"{BASE_URL}/en/{artist_slug}/all-works/text-list"
    soup = BeautifulSoup(fetch_wikiart(page_url).text, "html.parser")
    main = soup.find("main")
    if main is None:
        raise ValueError(f"Could not find the main content on {page_url}")

    artworks = []
    seen_urls = set()
    for link in main.find_all("a", href=True):
        artwork_url = urljoin(BASE_URL, link["href"])
        parts = [part for part in urlparse(artwork_url).path.split("/") if part]
        if (
            len(parts) != 3
            or parts[0].lower() != "en"
            or parts[1].lower() != artist_slug.lower()
            or artwork_url in seen_urls
        ):
            continue
        artwork_slug = parts[2]
        title = link.get_text(" ", strip=True) or artwork_slug.replace("-", " ").title()
        artworks.append((artwork_url, title, artwork_slug))
        seen_urls.add(artwork_url)
    return artworks

def get_artwork_metadata(artwork_url):
    """Read image, style, and medium fields from one artwork page."""
    soup = BeautifulSoup(fetch_wikiart(artwork_url).text, "html.parser")
    image_meta = soup.find("meta", attrs={"property": "og:image"})
    if image_meta is None or not image_meta.get("content"):
        raise ValueError(f"No og:image metadata found on {artwork_url}")

    values = {"style": [], "medium": []}
    field_names = {"style": "style", "styles": "style", "medium": "medium", "media": "medium"}
    # Inspect labeled list entries only, so unrelated links are not read as metadata.
    for item in soup.select("li.dictionary-values, li"):
        label_node = item.find(["s", "strong", "b"])
        if label_node is None:
            continue
        label = normalize_label(label_node.get_text(" ", strip=True).rstrip(":"))
        field = field_names.get(label)
        if field is None:
            continue
        linked_values = [
            link.get_text(" ", strip=True)
            for link in item.select("a")
            if link.get_text(" ", strip=True)
        ]
        label_text = label_node.get_text(" ", strip=True)
        plain_value = item.get_text(" ", strip=True).replace(label_text, "", 1).strip(" :")
        for value in linked_values or ([plain_value] if plain_value else []):
            if value and value not in values[field]:
                values[field].append(value)
    return urljoin(artwork_url, image_meta["content"]), values["style"], values["medium"]

def is_painting_medium(medium_values):
    """Return True when WikiArt's medium field matches a painting term."""
    combined = normalize_label(" ".join(medium_values))
    return bool(combined) and any(
        normalize_label(term) in combined for term in PAINTING_MEDIA_KEYWORDS
    )

def download_image(image_url, artist_slug, artwork_slug):
    """Download a WikiArt image and return its local path."""
    parsed = urlparse(image_url)
    host = (parsed.hostname or "").lower()
    if parsed.scheme != "https" or not host.endswith(".wikiart.org"):
        raise ValueError(f"Refusing a non-WikiArt image URL: {image_url}")
    response = fetch_wikiart(image_url)
    content_type = response.headers.get("Content-Type", "").lower()
    if not content_type.startswith("image/") or not response.content:
        raise ValueError(f"The image response was empty or not an image: {image_url}")

    extension = Path(parsed.path).suffix.lower()
    if not extension or len(extension) > 5:
        extension = ".jpg"
    image_path = IMAGE_FOLDER / f"{artist_slug}--{artwork_slug}{extension}"
    image_path.write_bytes(response.content)
    return image_path

### Collect the full dataset once

Aims for 100 accepted, unique paintings per class, stopping early for a class once it reaches 100. If WikiArt has too little usable records or requests fail, it reports the actual count and writes failures to `collection_errors.csv`; it does not run a separate catch-up scrape.

In [ ]:
# For each class, walk through the selected artists and keep collecting until
# the target is reached or the available artist/work pages are exhausted.

records = []
errors = []
seen_artwork_urls = set()
seen_image_urls = set()
style_rejected = 0
medium_rejected = 0

for movement, movement_artists in selected_artists.groupby("movement", sort=False):
    movement_count = 0

    for _, artist in movement_artists.iterrows():
        if movement_count >= TARGET_PER_MOVEMENT:
            break

        try:
            artwork_links = get_artwork_links(artist["artist_slug"])
        except (requests.RequestException, ValueError) as error:
            errors.append({
                "movement": movement,
                "artist_slug": artist["artist_slug"],
                "url": f"{BASE_URL}/en/{artist['artist_slug']}/all-works/text-list",
                "error": str(error),
            })
            continue

        successful_for_artist = 0
        attempts_for_artist = 0
        # Count only unique image downloads that pass both metadata filters.
        for artwork_url, title, artwork_slug in artwork_links:
            if movement_count >= TARGET_PER_MOVEMENT:
                break
            if successful_for_artist >= MAX_ARTWORKS_PER_ARTIST:
                break
            if attempts_for_artist >= MAX_ARTWORK_ATTEMPTS_PER_ARTIST:
                break
            if artwork_url in seen_artwork_urls:
                continue

            seen_artwork_urls.add(artwork_url)
            attempts_for_artist += 1
            try:
                image_url, artwork_styles, media = get_artwork_metadata(artwork_url)
                matching_styles = [
                    style for style in artwork_styles
                    if STYLE_TO_MOVEMENT.get(normalize_label(style)) == movement
                ]
                if not matching_styles:
                    style_rejected += 1
                    continue
                if not is_painting_medium(media):
                    medium_rejected += 1
                    continue
                if image_url in seen_image_urls:
                    continue
                image_path = download_image(image_url, artist["artist_slug"], artwork_slug)
            except (requests.RequestException, ValueError) as error:
                errors.append({
                    "movement": movement,
                    "artist_slug": artist["artist_slug"],
                    "url": artwork_url,
                    "error": str(error),
                })
                continue

            seen_image_urls.add(image_url)
            records.append({
                "title": title,
                "artist_name": artist["artist_name"],
                "artist_slug": artist["artist_slug"],
                "movement": movement,
                "artwork_styles": "; ".join(matching_styles),
                "medium": "; ".join(media),
                "artwork_url": artwork_url,
                "image_url": image_url,
                "image_path": str(image_path.relative_to(OUTPUT_FOLDER)),
            })
            successful_for_artist += 1
            movement_count += 1

        print(
            f"{movement}: {movement_count}/{TARGET_PER_MOVEMENT} paintings "
            f"after {artist['artist_name']} "
            f"({successful_for_artist} saved, {attempts_for_artist} checked)"
        )

if not records:
    raise RuntimeError(
        "No paintings passed the Style and medium filters. Inspect WikiArt's "
        "metadata and adjust the parser or PAINTING_MEDIA_KEYWORDS."
    )

# Save the one-pass results and any request errors for review.
paintings = pd.DataFrame(records).drop_duplicates(subset=["artwork_url", "image_url"])
paintings_path = OUTPUT_FOLDER / "wikiart_paintings.csv"
paintings.to_csv(paintings_path, index=False)
if errors:
    errors_path = OUTPUT_FOLDER / "collection_errors.csv"
    pd.DataFrame(errors).to_csv(errors_path, index=False)
    print(f"Saved {len(errors)} request errors to {errors_path}")

print(f"Saved {len(paintings)} painting records to {paintings_path}")
print("Collected records by movement:")
print(paintings.groupby("movement").size())
print(f"Skipped for missing/mismatched style: {style_rejected}")
print(f"Skipped for missing/unaccepted medium: {medium_rejected}")

### Package and download results

Create one ZIP containing the images referenced by the dataset CSV, then download that archive together with `artists.csv` and `wikiart_paintings.csv`.

In [ ]:
# Build a ZIP containing only the image files referenced by the CSV, then
# download the artist list, painting metadata, and image archive from Colab.

from zipfile import ZIP_DEFLATED, ZipFile

image_zip_path = OUTPUT_FOLDER / "wikiart_images.zip"
with ZipFile(image_zip_path, "w", compression=ZIP_DEFLATED) as archive:
    for relative_path in paintings["image_path"].drop_duplicates():
        image_path = OUTPUT_FOLDER / relative_path
        if image_path.exists():
            archive.write(image_path, arcname=relative_path)

files.download(str(artists_path))
files.download(str(paintings_path))
files.download(str(image_zip_path))